# Linear-Regression V2 vs V5 — head-to-head comparison

Both scripts read the **same dataset** (`FRED_all.xlsx`), predict the **same label** (`LABEL_Next_Month_Inflation`) and use the **same algorithm** (OLS Linear Regression). They differ only in two well-defined ways:

| | `linearregressionV2.py` | `LinearRegressionDateLimitV5.py` |
|---|---|---|
| Features | `Date_Unix, Year, Month` + every other numeric column | `Year, Month` + every other numeric column (no `Date_Unix`) |
| Train/test split | random 80/20 (`seed=42`) | chronological: train `< 2022-12-01`, test `2022-12-01 – 2025-12-31`, with Sep-2023 and Nov-2023 dropped |
| Row filtering | `dropna` on features + label only | same + the date filter above |

Because everything else is identical, this notebook can compare them **directly**.

**Implementation note:** the original scripts use PySpark MLlib `LinearRegression`. This notebook uses scikit-learn `LinearRegression` for portability — the underlying OLS solution is mathematically identical, so RMSE / MAE / R² / coefficients match what Spark would return.

---
## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

EXCEL_PATH  = "FRED_all.xlsx"
DATE_COL    = "Unnamed: 0"        # the unnamed first column — same convention as the original scripts
LABEL_COL   = "LABEL_Next_Month_Inflation"
RANDOM_SEED = 42

---
## 2. Load and inspect FRED_all.xlsx

Sanity-check the schema before training. The `Unnamed: 0` column is parsed as a date (this is how both scripts treat it).

In [ ]:
df_raw = pd.read_excel(EXCEL_PATH)
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])

print(f"Shape       : {df_raw.shape}")
print(f"Date range  : {df_raw[DATE_COL].min().date()} → {df_raw[DATE_COL].max().date()}")
print(f"Label col   : {LABEL_COL!r}  (present: {LABEL_COL in df_raw.columns})")
print(f"\nColumns ({len(df_raw.columns)}): {df_raw.columns.tolist()[:25]}{' ...' if len(df_raw.columns) > 25 else ''}")
df_raw.head()

In [ ]:
missing_pct = df_raw.isnull().mean().mul(100).round(1).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct):
    print("Missing values (% of rows):")
    print(missing_pct.to_string())
else:
    print("No missing values.")

---
## 3. Helper functions and shared feature engineering

In [ ]:
def add_time_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Add Date_Unix / Year / Month columns just like the PySpark scripts."""
    out = df.copy()
    out["Date_Unix"] = (out[date_col].astype("int64") // 10**9)   # seconds since epoch
    out["Year"]      = out[date_col].dt.year
    out["Month"]     = out[date_col].dt.month
    return out


def regression_metrics(y_true, y_pred) -> dict:
    return {
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE":  mean_absolute_error(y_true, y_pred),
        "R2":   r2_score(y_true, y_pred),
    }


df_full = add_time_features(df_raw, DATE_COL)

base_features = [c for c in df_full.columns
                 if c not in (DATE_COL, LABEL_COL, "Date_Unix", "Year", "Month")
                 and pd.api.types.is_numeric_dtype(df_full[c])]

FEATURES_V2 = ["Date_Unix", "Year", "Month"] + base_features    # mirrors linearregressionV2.py
FEATURES_V5 = ["Year", "Month"]              + base_features    # mirrors LinearRegressionDateLimitV5.py

print(f"V2 features ({len(FEATURES_V2)}): {FEATURES_V2}")
print(f"V5 features ({len(FEATURES_V5)}): {FEATURES_V5}")

---
## 4. Model V2 — random 80/20 split

`linearregressionV2.py`: drop NaN rows on `features + label`, then `randomSplit([0.8, 0.2], seed=42)`. Features include `Date_Unix`.

In [ ]:
cols_needed = [DATE_COL] + FEATURES_V2 + [LABEL_COL]
df_v2 = df_full[cols_needed].dropna(subset=FEATURES_V2 + [LABEL_COL]).reset_index(drop=True)

X_tr, X_te, y_tr, y_te, d_tr, d_te = train_test_split(
    df_v2[FEATURES_V2], df_v2[LABEL_COL], df_v2[DATE_COL],
    test_size=0.2, random_state=RANDOM_SEED,
)

lr_v2 = LinearRegression().fit(X_tr, y_tr)
y_pred_v2 = lr_v2.predict(X_te)

metrics_v2 = regression_metrics(y_te, y_pred_v2)
print(f"V2  rows after dropna : {len(df_v2):,}")
print(f"V2  train rows        : {len(X_tr):,}")
print(f"V2  test rows         : {len(X_te):,}")
print(f"V2  intercept         : {lr_v2.intercept_:.6f}\n")
print("V2  test metrics:")
for k, v in metrics_v2.items():
    print(f"  {k:<5}: {v:.6f}")

preds_v2 = pd.DataFrame({
    DATE_COL:     d_te.values,
    "actual":     y_te.values,
    "predicted":  y_pred_v2,
    "residual":   y_te.values - y_pred_v2,
}).sort_values(DATE_COL).reset_index(drop=True)

---
## 5. Model V5 — date-limited chronological split

`LinearRegressionDateLimitV5.py`:
* drop NaN rows on `features + label`,
* drop Sep-2023 and Nov-2023,
* keep dates `≤ 2025-12-31`,
* train on `Date < 2022-12-01`, test on `2022-12-01 ≤ Date ≤ 2025-12-31`.

Features do **not** include `Date_Unix`.

In [ ]:
cols_needed = [DATE_COL] + FEATURES_V5 + [LABEL_COL]
df_v5 = df_full[cols_needed].dropna(subset=FEATURES_V5 + [LABEL_COL]).copy()

exclude_mask = (
    ((df_v5["Year"] == 2023) & (df_v5["Month"].isin([9, 11]))) |
    (df_v5[DATE_COL] > pd.Timestamp("2025-12-31"))
)
df_v5 = df_v5.loc[~exclude_mask].sort_values(DATE_COL).reset_index(drop=True)

train_mask = df_v5[DATE_COL] <  pd.Timestamp("2022-12-01")
test_mask  = (df_v5[DATE_COL] >= pd.Timestamp("2022-12-01")) & (df_v5[DATE_COL] <= pd.Timestamp("2025-12-31"))

X_tr = df_v5.loc[train_mask, FEATURES_V5]
y_tr = df_v5.loc[train_mask, LABEL_COL]
X_te = df_v5.loc[test_mask,  FEATURES_V5]
y_te = df_v5.loc[test_mask,  LABEL_COL]
d_te = df_v5.loc[test_mask,  DATE_COL]

lr_v5 = LinearRegression().fit(X_tr, y_tr)
y_pred_v5 = lr_v5.predict(X_te)

metrics_v5 = regression_metrics(y_te, y_pred_v5)
print(f"V5  rows after filtering : {len(df_v5):,}")
print(f"V5  train rows           : {len(X_tr):,}  ({df_v5.loc[train_mask, DATE_COL].min().date()} → {df_v5.loc[train_mask, DATE_COL].max().date()})")
print(f"V5  test rows            : {len(X_te):,}  ({d_te.min().date()} → {d_te.max().date()})")
print(f"V5  intercept            : {lr_v5.intercept_:.6f}\n")
print("V5  test metrics:")
for k, v in metrics_v5.items():
    print(f"  {k:<5}: {v:.6f}")

preds_v5 = pd.DataFrame({
    DATE_COL:     d_te.values,
    "actual":     y_te.values,
    "predicted":  y_pred_v5,
    "residual":   y_te.values - y_pred_v5,
}).sort_values(DATE_COL).reset_index(drop=True)

---
## 6. Side-by-side metrics

Each model is evaluated on its **own** test set first — V2's randomly-shuffled 20 % vs V5's chronological 2022–12 → 2025–12 window. 
Section 9 retrains both on the **same** test window for a fair head-to-head.

In [ ]:
summary = pd.DataFrame({
    "V2 — random 80/20": metrics_v2,
    "V5 — date-limited": metrics_v5,
}).T
summary["Test rows"]  = [len(preds_v2), len(preds_v5)]
summary["Test start"] = [preds_v2[DATE_COL].min().date(), preds_v5[DATE_COL].min().date()]
summary["Test end"]   = [preds_v2[DATE_COL].max().date(), preds_v5[DATE_COL].max().date()]
summary = summary[["Test rows", "Test start", "Test end", "RMSE", "MAE", "R2"]]
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metric_names = ["RMSE", "MAE", "R2"]
model_labels = list(summary.index)
colors       = ["#4C72B0", "#DD8452"]

for ax, m in zip(axes, metric_names):
    bars = ax.bar(model_labels, summary[m].values, color=colors, edgecolor="white")
    ax.set_title(m)
    ax.tick_params(axis="x", rotation=15)
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
    if m == "R2":
        ax.axhline(0, color="black", linewidth=0.6)

plt.suptitle("V2 vs V5 — each model on its own test set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Visualisations (each model on its own test set)

### 7.1 Predicted vs Actual over time

In [ ]:
panels = [
    ("V2 — random 80/20", preds_v2, "#4C72B0"),
    ("V5 — date-limited", preds_v5, "#DD8452"),
]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)
for ax, (title, pdf, color) in zip(axes, panels):
    ax.plot(pdf[DATE_COL], pdf["actual"],    label="Actual",    color="black", linewidth=1.4)
    ax.plot(pdf[DATE_COL], pdf["predicted"], label="Predicted", color=color,   linewidth=1.4, linestyle="--")
    ax.set_title(title)
    ax.set_ylabel(LABEL_COL)
    ax.legend(loc="upper left")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

### 7.2 Predicted vs Actual scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, pdf, color) in zip(axes, panels):
    ax.scatter(pdf["actual"], pdf["predicted"],
               alpha=0.6, s=22, color=color, edgecolors="white", linewidths=0.4)
    lims = [min(pdf["actual"].min(), pdf["predicted"].min()),
            max(pdf["actual"].max(), pdf["predicted"].max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)
    r2 = r2_score(pdf["actual"], pdf["predicted"])
    ax.text(0.05, 0.92, f"R² = {r2:.4f}", transform=ax.transAxes, fontsize=11)
    ax.set_title(title)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.show()

### 7.3 Residuals over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=False)
for ax, (title, pdf, _) in zip(axes, panels):
    cols = ["#C44E52" if v < 0 else "#55A868" for v in pdf["residual"]]
    ax.bar(pdf[DATE_COL], pdf["residual"], color=cols, width=20, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(f"{title}  —  residuals (actual − predicted)")
    ax.set_ylabel("Residual")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

### 7.4 Residual distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (title, pdf, color) in zip(axes, panels):
    sns.histplot(pdf["residual"], bins=30, kde=True, ax=ax, color=color, edgecolor="white")
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    mean_res = pdf["residual"].mean()
    std_res  = pdf["residual"].std()
    ax.text(0.05, 0.92, f"mean = {mean_res:.4f}\nstd  = {std_res:.4f}",
            transform=ax.transAxes, fontsize=10, va="top")
    ax.set_title(title)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 7.5 Coefficient comparison

Each model fits an intercept plus a weight per feature. Here we line them up side by side. 
`Date_Unix` is only used by V2, so V5's bar for that feature is left blank.

In [ ]:
all_features = list(dict.fromkeys(FEATURES_V2 + FEATURES_V5))
coef_v2 = pd.Series(lr_v2.coef_, index=FEATURES_V2).reindex(all_features)
coef_v5 = pd.Series(lr_v5.coef_, index=FEATURES_V5).reindex(all_features)

coef_df = pd.DataFrame({"V2": coef_v2, "V5": coef_v5})

fig, ax = plt.subplots(figsize=(11, 0.4 * len(all_features) + 1.5))
y = np.arange(len(all_features))
h = 0.4
ax.barh(y - h/2, coef_df["V2"].fillna(0), height=h, label="V2", color="#4C72B0", edgecolor="white")
ax.barh(y + h/2, coef_df["V5"].fillna(0), height=h, label="V5", color="#DD8452", edgecolor="white")
ax.set_yticks(y)
ax.set_yticklabels(all_features)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.7)
ax.set_title("Linear-Regression coefficients — V2 vs V5")
ax.set_xlabel("Coefficient (effect on next-month inflation per unit of feature)")
ax.legend()
plt.tight_layout()
plt.show()

coef_df.assign(intercept_v2=lr_v2.intercept_, intercept_v5=lr_v5.intercept_)

---
## 8. Fair head-to-head — same test window

The Section-6 metrics use **different test windows** (V2 = random 20 %, V5 = Dec-2022 → Dec-2025). 
To compare apples to apples, we now retrain V2 on V5's chronological train set and evaluate **both** models on V5's chronological test window.

In [ ]:
df_common = df_full.dropna(subset=FEATURES_V2 + [LABEL_COL]).copy()
exclude_mask = (
    ((df_common["Year"] == 2023) & (df_common["Month"].isin([9, 11]))) |
    (df_common[DATE_COL] > pd.Timestamp("2025-12-31"))
)
df_common = df_common.loc[~exclude_mask].sort_values(DATE_COL).reset_index(drop=True)

train_mask_c = df_common[DATE_COL] <  pd.Timestamp("2022-12-01")
test_mask_c  = (df_common[DATE_COL] >= pd.Timestamp("2022-12-01")) & (df_common[DATE_COL] <= pd.Timestamp("2025-12-31"))

y_tr_c   = df_common.loc[train_mask_c, LABEL_COL]
y_te_c   = df_common.loc[test_mask_c,  LABEL_COL]
d_te_c   = df_common.loc[test_mask_c,  DATE_COL]

lr_v2_fair = LinearRegression().fit(df_common.loc[train_mask_c, FEATURES_V2], y_tr_c)
lr_v5_fair = LinearRegression().fit(df_common.loc[train_mask_c, FEATURES_V5], y_tr_c)

y_p_v2_fair = lr_v2_fair.predict(df_common.loc[test_mask_c, FEATURES_V2])
y_p_v5_fair = lr_v5_fair.predict(df_common.loc[test_mask_c, FEATURES_V5])

fair_metrics = pd.DataFrame({
    "V2 (with Date_Unix)": regression_metrics(y_te_c, y_p_v2_fair),
    "V5 (no Date_Unix)":    regression_metrics(y_te_c, y_p_v5_fair),
}).T
fair_metrics["Test rows"]  = len(y_te_c)
fair_metrics["Test start"] = d_te_c.min().date()
fair_metrics["Test end"]   = d_te_c.max().date()
fair_metrics = fair_metrics[["Test rows", "Test start", "Test end", "RMSE", "MAE", "R2"]]
fair_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(d_te_c, y_te_c.values,  label="Actual",            color="black",   linewidth=1.6)
ax.plot(d_te_c, y_p_v2_fair,    label="V2 (with Date_Unix)", color="#4C72B0", linewidth=1.4, linestyle="--")
ax.plot(d_te_c, y_p_v5_fair,    label="V5 (no Date_Unix)",   color="#DD8452", linewidth=1.4, linestyle="--")
ax.set_title(f"Fair comparison — same chronological test window  ({d_te_c.min().date()} → {d_te_c.max().date()})")
ax.set_xlabel("Date")
ax.set_ylabel(LABEL_COL)
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Summary

* V2 and V5 share input data, label and algorithm — the comparison is **direct** and clean.
* The two real differences are:
  1. **Train/test split** (random shuffle vs chronological date filter).
  2. **Feature set** (V2 includes `Date_Unix`, V5 does not).
* Section 6 reports each model on its own test window. These are *not* the same window, so the metrics there reflect both the model **and** the difficulty of the period.
* Section 8 retrains both models on the **same** chronological train/test window (V5's), which isolates the effect of the `Date_Unix` feature alone.
* Coefficient differences are easy to read off Section 7.5 — large gaps usually appear because the random split (V2) leaks future information back into training, which can inflate or deflate certain coefficients.